# PHẦN VI — KOLLA-ANSIBLE DEPLOY (RUNBOOK RÚT GỌN)

Notebook rút gọn: chỉ lệnh + comment ngắn.

## BƯỚC 24 — Activate Kolla venv

In [ ]:
# Chạy trên Controller VM1
source /opt/kolla-venv/bin/activate
which kolla-ansible
kolla-ansible --version

## BƯỚC 24 — Chuẩn bị /etc/kolla

In [ ]:
sudo mkdir -p /etc/kolla
sudo chown -R ubuntu:ubuntu /etc/kolla
cp -r /opt/kolla-venv/share/kolla-ansible/etc_examples/kolla/* /etc/kolla/

ls -lah /etc/kolla

## BƯỚC 24 — Copy Ceph config vào Kolla

In [ ]:
sudo mkdir -p /etc/ceph
sudo mkdir -p /etc/kolla/config/cinder/cinder-volume
sudo mkdir -p /etc/kolla/config/nova
sudo mkdir -p /etc/kolla/config/glance

sudo cp /tmp/ceph.conf /etc/ceph/
sudo cp /tmp/ceph.client.*.keyring /etc/ceph/

sudo cp /etc/ceph/ceph.client.glance.keyring /etc/kolla/config/glance/
sudo cp /etc/ceph/ceph.conf /etc/kolla/config/glance/

sudo cp /etc/ceph/ceph.client.cinder.keyring /etc/kolla/config/cinder/cinder-volume/
sudo cp /etc/ceph/ceph.conf /etc/kolla/config/cinder/cinder-volume/

sudo cp /etc/ceph/ceph.client.nova.keyring /etc/kolla/config/nova/
sudo cp /etc/ceph/ceph.conf /etc/kolla/config/nova/

find /etc/kolla/config -type f -name "ceph*" -exec ls -lh {} \;

## BƯỚC 25 — Tạo multinode inventory

In [ ]:
cat > /etc/kolla/multinode << EOF
[control]
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[network]
# OVN distributed

[compute]
10.0.0.21 ansible_user=stack
10.0.0.22 ansible_user=stack
10.0.0.23 ansible_user=stack

[storage]
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[monitoring]
10.0.0.11 ansible_user=ubuntu

[deployment]
localhost ansible_connection=local
EOF

cat /etc/kolla/multinode

## BƯỚC 25 — SSH key + Ansible ping

In [ ]:
ssh-keygen -t ed25519 -N "" -f ~/.ssh/id_ed25519

for ip in 10.0.0.11 10.0.0.12 10.0.0.13; do
  ssh-copy-id -o StrictHostKeyChecking=no ubuntu@$ip
done

for ip in 10.0.0.21 10.0.0.22 10.0.0.23; do
  ssh-copy-id -o StrictHostKeyChecking=no stack@$ip
done

source /opt/kolla-venv/bin/activate
ansible -i /etc/kolla/multinode all -m ping

## BƯỚC 26 — Kiểm tra bridge toàn cụm

In [ ]:
source /opt/kolla-venv/bin/activate

ansible -i /etc/kolla/multinode all -m shell -a "ip -br addr"
ansible -i /etc/kolla/multinode all -m shell -a "ip link show br-mgmt || true"
ansible -i /etc/kolla/multinode all -m shell -a "ip link show br-storage || true"
ansible -i /etc/kolla/multinode all -m shell -a "ip link show br-ex || true" 

## BƯỚC 26 — globals.yml

In [ ]:
cat > /etc/kolla/globals.yml << EOF
kolla_base_distro: ubuntu
kolla_install_type: binary
openstack_release: "2025.1"

network_interface: "br-mgmt"
neutron_external_interface: "br-ex"
kolla_internal_vip_address: "10.0.0.10"
kolla_external_vip_address: "192.168.150.200"

storage_interface: "br-storage"
tunnel_interface: "br-mgmt"

enable_openstack_core: "yes"
enable_glance: "yes"
enable_nova: "yes"
enable_neutron: "yes"
enable_cinder: "yes"
enable_horizon: "yes"
enable_heat: "no"
enable_swift: "no"

neutron_plugin_agent: "ovn"
neutron_ovn_distributed_fip: "yes"

glance_backend_ceph: "yes"
glance_backend_file: "no"
cinder_backend_ceph: "yes"
nova_backend_ceph: "yes"

ceph_glance_user: "glance"
ceph_glance_pool_name: "images"
ceph_cinder_user: "cinder"
ceph_cinder_pool_name: "volumes"
ceph_nova_user: "nova"
ceph_nova_pool_name: "vms"

ceph_mon_nodes: "10.0.0.21,10.0.0.22,10.0.0.23"
EOF

cat /etc/kolla/globals.yml

## BƯỚC 26 — Check VIP/Ceph reachability

In [ ]:
ping -c 2 10.0.0.10 || true
ping -c 2 192.168.150.200 || true

for ip in 10.0.0.21 10.0.0.22 10.0.0.23; do
  ping -c 3 $ip
done

ceph -c /etc/ceph/ceph.conf -s || true

## BƯỚC 27 — tmux + password

In [ ]:
tmux new -s kolla-deploy

# Trong tmux:
source /opt/kolla-venv/bin/activate
cd /etc/kolla

cp /etc/kolla/passwords.yml /etc/kolla/passwords.yml.bak.$(date +%F-%H%M%S) || true
kolla-genpwd

sed -i "s/^keystone_admin_password:.*/keystone_admin_password: 123.abc/" /etc/kolla/passwords.yml
grep "^keystone_admin_password:" /etc/kolla/passwords.yml

## BƯỚC 27 — Bootstrap + prechecks

In [ ]:
source /opt/kolla-venv/bin/activate

kolla-ansible -i /etc/kolla/multinode bootstrap-servers

ansible -i /etc/kolla/multinode all -m shell -a "docker --version || true"
ansible -i /etc/kolla/multinode all -m shell -a "systemctl is-active docker || true"

kolla-ansible -i /etc/kolla/multinode prechecks  # phải 0 FAILED

## BƯỚC 27 — Pull + deploy + post-deploy

In [ ]:
source /opt/kolla-venv/bin/activate

kolla-ansible -i /etc/kolla/multinode pull
kolla-ansible -i /etc/kolla/multinode deploy
kolla-ansible -i /etc/kolla/multinode post-deploy

ls -lh /etc/kolla/admin-openrc.sh

## BƯỚC 28 — Verify OpenStack

In [ ]:
source /opt/kolla-venv/bin/activate
which openstack || pip install python-openstackclient
source /etc/kolla/admin-openrc.sh

openstack token issue
openstack service list
openstack endpoint list
openstack compute service list
openstack volume service list
openstack hypervisor list

## BƯỚC 28 — Verify containers

In [ ]:
docker ps
docker ps -a --filter status=exited

ansible -i /etc/kolla/multinode all -m shell -a "docker ps --format '{{.Names}}' | head -20"
ansible -i /etc/kolla/multinode all -m shell -a "docker ps -a --filter status=exited --format '{{.Names}}'" 